In [8]:
data = [
    (101,"Rahul",75000),
    (102,"Priya",85000),
    (103,"Amit",65000)
]

df = spark.createDataFrame(data, ["emp_id","name","salary"])
df.write.format("delta").mode("overwrite").save("/tmp/employees_delta")
delta_df = spark.read.format("delta").load("/tmp/employees_delta")
display(delta_df.show())

+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|   102|Priya| 85000|
|   103| Amit| 65000|
|   101|Rahul| 75000|
+------+-----+------+



None

In [9]:
df.write.format("delta").mode("overwrite").saveAsTable("employees")

In [10]:
spark.sql("SELECT * FROM employees").show()

+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|   102|Priya| 85000|
|   103| Amit| 65000|
|   101|Rahul| 75000|
+------+-----+------+



In [16]:
spark.sql("""
CREATE TABLE employees_delta (
    emp_id INT,
    name STRING,
    salary INT
)
USING DELTA
""")

DataFrame[]

In [17]:
spark.sql("""
INSERT INTO employees_delta VALUES
(101,'Rahul',75000),
(102,'Priya',85000)
""")

DataFrame[]

In [18]:
updates = [
    (102,"Priya",90000),
    (104,"Sneha",70000)
]

updates_df = spark.createDataFrame(updates, ["emp_id","name","salary"])

In [19]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/tmp/employees_delta")

delta_table.alias("target") \
.merge(
    updates_df.alias("source"),
    "target.emp_id = source.emp_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [20]:
spark.read.format("delta").load("/tmp/employees_delta").show()

+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|   102|Priya| 90000|
|   103| Amit| 65000|
|   104|Sneha| 70000|
|   101|Rahul| 75000|
+------+-----+------+



In [21]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, "/tmp/employees_delta")

delta_table.history().show(truncate=False)

+-------+-----------------------+------+--------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------------

In [22]:
spark.sql("DESCRIBE HISTORY delta.`/tmp/employees_delta`").show(truncate=False)

+-------+-----------------------+------+--------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-------------

In [24]:
df_version_1 = spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .load("/tmp/employees_delta")

display(df_version_1.show())

+------+-----+------+
|emp_id| name|salary|
+------+-----+------+
|   102|Priya| 85000|
|   103| Amit| 65000|
|   101|Rahul| 75000|
+------+-----+------+



None